# Matplotlib 之五：箱型图（boxplot）

---

箱型图是看**数据分布**的神器：不用画出每一个点，就能把"中位数、四分位、范围、异常值"一股脑展示出来。用来**比较多组数据的分布差异**最合适。

## 学习目标

1. 用 `plt.boxplot()` 画箱型图，理解箱子里每一根线代表什么
2. 看懂"中位数 / 四分位 / 须线 / 异常值(flier)"
3. 会用 `patch_artist`、各种 `*props` 给箱子换颜色
4. 知道箱型图适合回答"几组数据的分布/离散程度差别大不大"
5. 再次巩固中文显示配置


---
## 一、什么时候用箱型图？

| 图 | 回答的问题 |
|---|---|
| 柱状图 | 各类别"平均值"谁高 |
| 散点图 | 两变量是否相关 |
| **箱型图** | 一组数据"分布长啥样、离不离谱、有没有异常值" |

如果你只想比"平均"，柱状图够；但如果你想比"分布是否稳定、有没有极端值"，**箱型图比柱状图信息量大得多**。


## 二、箱子里的每一根线是什么意思（五数概括）

```
        |      <- 上须线：最大值（须线只到 1.5×IQR 内）
    ┌───┴───┐
    │   ────│      <- 中位数(median)
    └───┬───┘
        |      <- 下须线：最小值
        ○      <- 异常值(flier)：超出须线范围的离群点
```

| 部位 | 含义 |
|---|---|
| 箱子顶边 | 上四分位数 Q3（75% 的数据在这之下） |
| 箱子底边 | 下四分位数 Q1（25% 的数据在这之下） |
| 箱子中线 | 中位数（不是平均值！） |
| 箱子高度 | IQR = Q3 - Q1，代表中间 50% 数据的离散程度 |
| 须线 | 通常到 `Q1-1.5×IQR` ~ `Q3+1.5×IQR` |
| 小圆圈 ○ | 异常值（超出须线范围的离群点） |


---
## 三、最基础的箱型图

`plt.boxplot(data)` 的 `data` 是一个"装着多组数组"的可迭代对象（如列表的列表、字典的值）。`tick_labels` 给每组起名（matplotlib ≥ 3.9 支持；本环境 3.11 没问题）。

> **原稿问题**：原稿标题写"随机数据的箱型图"，但数据其实是**手写硬编码**的，并不是随机生成的。标题与内容不符，下面根据实际数据改成"三门成绩分布箱型图"。另外原稿在 `boxplot` 前挂了一整段 `"""..."""` 解释字体查找顺序——那是代码注释，挪进 markdown 讲解更合适。


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 三门课的成绩（每组一组数），用 dict 顺便给每组命名
score = {"语文": [80, 85, 90, 95, 100, 93, 72, 82],
         "数学": [70, 75, 80, 85, 90, 88, 75, 82],
         "英语": [60, 65, 70, 75, 80, 85, 72, 82]}

plt.boxplot(score.values(),
            tick_labels=score.keys(),        # 每组名字显示在 x 轴
            patch_artist=True,               # 【关键】允许填充箱体颜色
            boxprops=dict(facecolor='lightblue', color='blue'),
            whiskerprops=dict(color='blue'),
            capprops=dict(color='blue'),
            medianprops=dict(color='red'),    # 中位数线标红，最醒目
            flierprops=dict(marker='o', markerfacecolor='red',
                            markersize=8, linestyle='none'))

plt.title("三门成绩分布箱型图")
plt.xlabel("科目")
plt.ylabel("分数")
plt.grid(True, axis='y', linestyle="--", alpha=0.5)
plt.show()   # 输出: 3 个蓝箱子+红中位数线，语文有个低分异常点


**怎么读这张图**：每个箱子的红线是中位数；箱子越高说明中间 50% 学生分差越大；"语文"箱子下方那个红圈，就是 72 分这个相对偏低、被算法判定为异常值的点。


---
## 四、横向箱型图与多组数据美化

加 `vert=False` 把箱子横过来（类别名很长时更好看）。再演示一次"多组数据 + 配色"。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 用 numpy 现场造 3 组不同分布的数据（均值/离散度不同）
np.random.seed(0)
g1 = np.random.normal(70, 5, 50)
g2 = np.random.normal(80, 10, 50)
g3 = np.random.normal(90, 3, 50)

plt.figure(figsize=(8, 6))
plt.boxplot([g1, g2, g3],
            tick_labels=['A组', 'B组', 'C组'],
            vert=False,                 # 横向
            patch_artist=True,
            boxprops=dict(facecolor='#99FF99', color='green'),
            medianprops=dict(color='red'),
            flierprops=dict(marker='x', markerfacecolor='red', markersize=8))
plt.title("三组数据分布对比（横向）")
plt.xlabel("数值")
plt.grid(True, axis='x', linestyle="--", alpha=0.5)
plt.show()   # 输出: 横着的 3 个绿箱子，C组最集中、B组最散


---
## 五、中文显示问题（再强调）

箱型图的 `tick_labels`、`title`、`xlabel` 带中文仍会变方块，配置照旧：

```python
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
```

Windows 用 `SimHei`/`Microsoft YaHei`；macOS 用 `Arial Unicode MS`；Linux 用 `WenQuanYi Zen Hei`。


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 中位数当平均值 | 误读分布 | 箱子中线是中位数，不是均值 |
| 箱子高=数值大 | 误以为组更高就更好 | 箱子高代表"离散度大"，不是"数值大" |
| 中文方块 | 科目名变 □ | 设 `font.sans-serif` 中文字体 |
| 箱子不填色 | 默认只有边框 | 加 `patch_artist=True` + `boxprops` |
| 标题与数据不符 | 误导读者 | 标题要如实描述数据来源 |

---

## 七、动手练习

**练习 1**：用 numpy 造两组正态分布数据（如均值 60/标准差 8，均值 75/标准差 5），各 40 个，画箱型图比较。

**练习 2**：给练习 1 的箱子填充颜色，并把中位数线标红。

**练习 3**：用 `np.random.randint` 造 3 组 0~100 的整数成绩，画横向箱型图。

**练习 4**：观察练习 3，找出哪组 IQR 最大（箱子最高），说明它离散度最大。

---

**答案思路（先自己做）**

```python
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(1)
a = np.random.normal(60, 8, 40)
b = np.random.normal(75, 5, 40)
c = np.random.randint(0, 101, 40)

plt.figure()
plt.boxplot([a, b, c], tick_labels=['A','B','C'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
            medianprops=dict(color='red'))
plt.show()
```
